In [2]:
!pip install -q vllm
!pip uninstall -y torchaudio

Found existing installation: torchaudio 2.11.0
Uninstalling torchaudio-2.11.0:
  Successfully uninstalled torchaudio-2.11.0


In [3]:
!git clone https://github.com/VatsalRoy/keep-it-warm.git
%cd /kaggle/working/keep-it-warm
!python -m env.generate_world --seed 42

Cloning into 'keep-it-warm'...
remote: Enumerating objects: 79, done.
remote: Counting objects: 100% (79/79), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 79 (delta 18), reused 68 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (79/79), 47.24 KiB | 6.75 MiB/s, done.
Resolving deltas: 100% (18/18), done.
/kaggle/working/keep-it-warm
seed=42  world_sha256=4a5b9a1a8c73a5e39f83a7c00eb5d4ccafec0d2bf6912ce82e81f31f2f0dc419


In [4]:
import os, subprocess

cmd = [
    "vllm", "serve", "Qwen/Qwen3-4B-Instruct-2507",
    "--dtype", "half",
    "--max-model-len", "16384",
    "--enable-prefix-caching",
    "--enable-prompt-tokens-details",
    "--gpu-memory-utilization", "0.90",
    "--port", "8000",
]
env = {**os.environ, "VLLM_SERVER_DEV_MODE": "1"}   # unlocks /reset_prefix_cache
log = open("vllm.log", "w")
server = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)
print("started, pid", server.pid)

started, pid 176


In [5]:
import time, requests
for i in range(90):
    if server.poll() is not None:
        print("Server CRASHED - see log below"); break
    try:
        if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
            print(f"READY after ~{i*10}s"); break
    except Exception:
        pass
    time.sleep(10)
else:
    print("Not ready after 15 min - see log below")
!tail -n 15 vllm.log

READY after ~200s
(APIServer pid=176) INFO 10-31 15:12:54 [launcher.py:195] Route: /generative_scoring, Methods: POST
(APIServer pid=176) INFO 10-31 15:12:54 [launcher.py:195] Route: /scale_elastic_ep, Methods: POST
(APIServer pid=176) INFO 10-31 15:12:54 [launcher.py:195] Route: /is_scaling_elastic_ep, Methods: POST
(APIServer pid=176) INFO 10-31 15:12:54 [launcher.py:195] Route: /v1/chat/completions/render, Methods: POST
(APIServer pid=176) INFO 10-31 15:12:54 [launcher.py:195] Route: /v1/messages/render, Methods: POST
(APIServer pid=176) INFO 10-31 15:12:54 [launcher.py:195] Route: /v1/completions/render, Methods: POST
(APIServer pid=176) INFO 10-31 15:12:54 [launcher.py:195] Route: /v1/responses/render, Methods: POST
(APIServer pid=176) INFO 10-31 15:12:54 [launcher.py:195] Route: /v1/chat/completions/derender, Methods: POST
(APIServer pid=176) INFO 10-31 15:12:54 [launcher.py:195] Route: /v1/completions/derender, Methods: POST
(APIServer pid=176) INFO 10-31 15:12:54 [launcher.py:1

In [6]:
print("reset_prefix_cache ->", requests.post("http://localhost:8000/reset_prefix_cache").status_code)
!grep -iE "KV cache|maximum concurrency" vllm.log

reset_prefix_cache -> 200
(EngineCore pid=253) INFO 10-31 15:11:16 [utils.py:347] Using LBNHC KV cache layout.
(EngineCore pid=253) INFO 10-31 15:12:16 [gpu_worker.py:759] Available KV cache memory: 4.08 GiB
(EngineCore pid=253) INFO 10-31 15:12:16 [gpu_worker.py:774] CUDA graph memory profiling is enabled (default since v0.21.0). The current --gpu-memory-utilization=0.9000 is equivalent to --gpu-memory-utilization=0.8713 without CUDA graph memory profiling. To maintain the same effective KV cache size as before, increase --gpu-memory-utilization to 0.9287. To disable, set VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0.
(EngineCore pid=253) INFO 10-31 15:12:16 [kv_cache_utils.py:2327] GPU KV cache size: 29,728 tokens, Maximum concurrency for 16,384 tokens per request: 1.81x
(EngineCore pid=253) INFO 10-31 15:12:50 [gpu_worker.py:1034] Free memory on device (14.46/14.56 GiB) on startup. Desired GPU memory utilization is (0.9, 13.11 GiB). Actual usage is 7.87 GiB for consumed memory (weights

In [7]:
!python scripts/run_episodes.py --policy full \
    --tasks t1-errcount-1,t1-most5xx-1,t2-oom-2,t2-config-1,t3-rootcause-1 \
    --out runs/baseline_smoke

BAD t1-most5xx-1     s0 turns=10  end=submitted        peak_ctx=981    cached=90% $B=0.0171 wall=18.6s  answer='none'
OK  t1-errcount-1    s0 turns=2   end=submitted        peak_ctx=652    cached=38% $B=0.0052 wall=3.4s  answer='4'
BAD t2-oom-2         s0 turns=13  end=submitted        peak_ctx=5220   cached=84% $B=0.0555 wall=51.3s  answer='unknown,jonas.berg'
BAD t2-config-1      s0 turns=13  end=submitted        peak_ctx=2215   cached=88% $B=0.0328 wall=33.7s  answer='unknown, unknown, unknown'
BAD t3-rootcause-1   s0 turns=12  end=submitted        peak_ctx=5950   cached=72% $B=0.0511 wall=47.4s  answer='search, d-1033, yuki.tanaka'


In [8]:
!python scripts/summarize_run.py runs/baseline_smoke --episode t3-rootcause-1

task             ok   turns end              peak_ctx  cached       $A       $B  naive$B  wall_s
t1-most5xx-1     NO      10 submitted             981     90%   0.0008   0.0171   0.0436    18.6
t1-errcount-1    yes      2 submitted             652     38%   0.0002   0.0052   0.0068     3.4
t2-oom-2         NO      13 submitted            5220     84%   0.0030   0.0555   0.1641    51.3
t2-config-1      NO      13 submitted            2215     88%   0.0016   0.0328   0.0886    33.7
t3-rootcause-1   NO      12 submitted            5950     72%   0.0023   0.0511   0.1127    47.4

5 episodes | accuracy 20% | mean turns 10.0 | mean peak context 3,004 tokens | cached 81%
cost: sheet A $0.0078, sheet B $0.1616 (naive B, ignoring cache: $0.4159)
GPU wall time 2.6 min total, 31 s per episode on average

prefix edits (first_changed < previous length): 0 of 45 turns
unparseable replies: 0

t3-rootcause-1: turn | prompt | cached | uncached | first_changed/prev | ttft_ms | action
    1 |    367 |   

In [9]:
import json
eps = [json.loads(l) for l in open("runs/baseline_smoke/episodes.jsonl")]
ep = next(e for e in eps if e["task_id"] == "t3-rootcause-1")
for m in ep["transcript"][2:]:          # skip system prompt + question
    if m["kind"] == "action":
        print("MODEL:", m["content"][:300])
    else:
        print("   TOOL:", m["content"][:120].replace("\n", " | "), "...")

MODEL: {"thought": "locate logs for the request failure on the specified date", "tool": "grep", "args": {"pattern": "r-3227", "path": "/logs/access.log"}}
   TOOL: [grep result] | error: no such file or directory: /logs/access.log ...
MODEL: {"thought": "check for logs in a different directory, such as /logs/app/", "tool": "grep", "args": {"pattern": "r-3227", "path": "/logs/app/"}}

   TOOL: [grep result] | error: no such file or directory: /logs/app/ ...
MODEL: {"thought": "check the root directory for relevant log files or service directories", "tool": "ls", "args": {"path": "/logs"}}
   TOOL: [ls result] | /logs: | api-gateway/ | auth/ | inventory/ | notifications/ | orders/ | payments/ | search/ | shipping/ ...
MODEL: {"thought": "search for r-3227 in the api-gateway logs as it might be a common entry point", "tool": "grep", "args": {"pattern": "r-3227", "path": "/logs/api-gateway/"}}

   TOOL: [grep result] | error: /logs/api-gateway/ is a directory; grep takes a single file ...
